## Stage 5 — STL Export

Converts the SIMP density field to a Fusion-360-quality watertight STL.

**Rust voxel path** (USE_RUST_SOLVER=True): reads `base_part_density.npy` (nz×ny×nx f32), runs skimage marching cubes, trimesh repair + Taubin smoothing + quadric decimation.

**FEniCSx tet path**: extracts the isosurface boundary from the DG0 density on the tet mesh.

Cell 0 — Parameters (tag: parameters)

In [ ]:
# Cell 0 — tagged: parameters
import os
os.chdir("/workspace")

import sys
sys.path.insert(0, "/workspace")

PART_NAME_OVERRIDE = None   # set by pipeline_full.ipynb in sweep mode
STAGE04_HANDOFF    = None   # auto-detect; Cell 1 picks up most recent *_stage04.json

# ── Isosurface threshold ──────────────────────────────────────────────────
# 0.45: default, more material retained, slightly rougher surface
# 0.48: cleaner surface, ~5% less material, recommended if grey% is low
DENSITY_THRESHOLD  = 0.50

# ── Smoothing ─────────────────────────────────────────────────────────────
SMOOTH_ITERATIONS  = 20     # Taubin passes (0 = raw MC, 10 = light, 30 = smooth)
TAUBIN_LAMB        = 0.5
TAUBIN_NU          = -0.53  # negative in standard Taubin formulation
SMOOTH_LAMBDA = 0.6

# ── Simplification ────────────────────────────────────────────────────────
TARGET_FACE_FRACTION = 0.30  # keep 10% of raw MC faces (target ~10k-50k)
MIN_FACES            = 5_000 # floor regardless of fraction

# ── Minimum feature size (morphological opening) ──────────────────────────
# 2 × nozzle diameter is a safe floor. 0.4mm nozzle → set 0.8 minimum,
# 1.6mm recommended for clean topology. Set 0.0 to disable.
MIN_FEATURE_MM = 1.6

# ── Island removal ────────────────────────────────────────────────────────
REMOVE_ISLANDS     = True
ISLAND_VOL_FRAC    = 0.05

RENDER_PLOTS = False


Cell 1 — Load Stage 4 handoff

In [ ]:
# Cell 1 — Resolve paths from stage04 handoff
from pathlib import Path
import json
import numpy as np

# ── Locate Stage 4 handoff ────────────────────────────────────────────────
if STAGE04_HANDOFF is not None:
    handoff_path = Path(STAGE04_HANDOFF)
elif PART_NAME_OVERRIDE is not None:
    handoff_path = Path("outputs/meshes") / f"{PART_NAME_OVERRIDE}_stage04.json"
else:
    candidates = sorted(Path("outputs/meshes").glob("*_stage04.json"),
                        key=lambda p: p.stat().st_mtime)
    assert candidates, \
        "No *_stage04.json found in outputs/meshes/ — run notebook 04 first."
    handoff_path = candidates[-1]
assert handoff_path.exists(), f"Stage04 handoff not found: {handoff_path}"

assert handoff_path.exists(), f"Stage04 handoff not found: {handoff_path}"
handoff = json.loads(handoff_path.read_text())

# ── Extract metadata ──────────────────────────────────────────────────────
solver_type = handoff.get("solver", "rust_voxel")
grid_config = handoff["grid"]          # KeyError is intentional: grid is required
part_name   = PART_NAME_OVERRIDE if PART_NAME_OVERRIDE else handoff["part_name"]

print(f"Stage04 handoff: {handoff_path.name}")
print(f"  Part:      {part_name}")
print(f"  Solver:    {solver_type}")
print(f"  Grid:      {grid_config['nx']}×{grid_config['ny']}×{grid_config['nz']}")
print(f"  Converged: {handoff.get('converged')}  "
      f"Iters: {handoff.get('n_iterations')}  "
      f"C_final: {handoff.get('final_compliance', 0):.4e}")

Stage04 handoff: cantilever_arm_stage04.json
  Part:      cantilever_arm
  Solver:    rust_voxel
  Grid:      100×50×75
  Converged: True  Iters: 251  C_final: 1.8270e-01


Cell 2 — Load density field and inspect polarization

In [3]:
# Cell 2 — Load density + checkpoint
# ──────────────────────────────────────────────────────────────────────────
# CHECKPOINT: inspect before proceeding.
# A good two-stage run should show a strongly bimodal histogram.
# If you see a broad unimodal hump, re-run notebook 04 with the two-stage
# continuation (Stage1: p=2 → Stage2: p=3 warm-started from Stage1).
# ──────────────────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt

if solver_type == "rust_voxel":
    nx = grid_config["nx"]
    ny = grid_config["ny"]
    nz = grid_config["nz"]
    voxel_m = grid_config["voxel_size"]

    # Try .npy first (saved by notebook 04), fall back to raw .bin
    npy_path = Path("outputs/meshes") / f"{part_name}_density.npy"
    bin_path = Path("outputs/problem/density.bin")

    if npy_path.exists():
        density = np.load(npy_path).astype(np.float32)  # (nz, ny, nx)
        print(f"Loaded density from {npy_path}")
    elif bin_path.exists():
        density = np.fromfile(bin_path, dtype=np.float32).reshape(nz, ny, nx)
        print(f"Loaded density from {bin_path}")
    else:
        raise FileNotFoundError(
            f"Cannot find density: {npy_path} or {bin_path}\n"
            "Run notebook 04 first."
        )

    assert density.shape == (nz, ny, nx), \
        f"Expected ({nz},{ny},{nx}), got {density.shape}"

    rho = density.ravel()
    frac_solid = (rho > 0.9).mean()
    frac_void  = (rho < 0.1).mean()
    frac_grey  = 1.0 - frac_solid - frac_void

    print(f"\nDensity stats:")
    print(f"  Shape:        {density.shape}  [{nx*voxel_m*1000:.0f}×{ny*voxel_m*1000:.0f}×{nz*voxel_m*1000:.0f}mm]")
    print(f"  Min/Max/Mean: {rho.min():.4f} / {rho.max():.4f} / {rho.mean():.4f}")
    print(f"  ρ > 0.9 (solid): {frac_solid*100:.1f}%")
    print(f"  ρ < 0.1 (void):  {frac_void*100:.1f}%")
    print(f"  Grey (0.1–0.9):  {frac_grey*100:.1f}%   ← target: < 20%")

    if frac_grey > 0.30:
        print("\n  ⚠ WARNING: >30% grey elements — density not well-polarized.")
        print("    Re-run notebook 04 with two-stage penal continuation (p=2 → p=3).")
    elif frac_grey > 0.20:
        print("\n  ℹ NOTE: 20-30% grey — moderate polarization. Results usable.")
    else:
        print("\n  ✓ Polarization looks good.")

    # Histogram
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(rho, bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[0].axvline(DENSITY_THRESHOLD, color='red', linestyle='--',
                    label=f'threshold={DENSITY_THRESHOLD}')
    axes[0].set_xlabel('Density ρ')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Density histogram — target: bimodal peaks at 0 and 1')
    axes[0].legend()

    # Mid-Z slice
    iz_mid = nz // 2
    im = axes[1].imshow(density[iz_mid], vmin=0, vmax=1, cmap='bone',
                        origin='lower', aspect='auto')
    axes[1].set_title(f'Mid-Z slice (iz={iz_mid})')
    axes[1].set_xlabel('ix (x-axis)')
    axes[1].set_ylabel('iy (y-axis)')
    plt.colorbar(im, ax=axes[1])

    plt.tight_layout()
    density_hist_path = Path('outputs/reports') / f'{part_name}_density_histogram.png'
    plt.savefig(density_hist_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"\nHistogram saved: {density_hist_path}")

else:
    # FEniCSx path — load from XDMF
    import h5py
    from mpi4py import MPI
    from dolfinx.io import XDMFFile

    xdmf_path    = Path("outputs/meshes/opt_domain.xdmf")
    density_path = Path(handoff.get('density_path',
                        f'outputs/meshes/{part_name}_density.xdmf'))
    assert xdmf_path.exists(), f"Not found: {xdmf_path}"
    assert density_path.exists(), f"Not found: {density_path}"

    comm = MPI.COMM_WORLD
    with XDMFFile(comm, str(xdmf_path), "r") as xdmf:
        domain = xdmf.read_mesh(name="Grid")
    coord_max = domain.geometry.x.max()
    if coord_max > 1.0:
        domain.geometry.x[:] /= 1000.0

    h5_path = str(density_path).replace('.xdmf', '.h5')
    with h5py.File(h5_path, 'r') as f:
        rho_values = f['Function/density/0'][:].ravel()

    pts_mm = domain.geometry.x * 1000.0
    tdim = domain.topology.dim
    domain.topology.create_connectivity(tdim, 0)
    conn   = domain.topology.connectivity(tdim, 0)
    n_elem = domain.topology.index_map(tdim).size_local
    tets   = np.array([conn.links(i) for i in range(n_elem)])
    print(f"FEniCSx mesh: {len(pts_mm):,} nodes, {n_elem:,} elements")
    print(f"Density range: [{rho_values.min():.4f}, {rho_values.max():.4f}]")


Loaded density from outputs/meshes/cantilever_arm_density.npy

Density stats:
  Shape:        (75, 50, 100)  [100×50×75mm]
  Min/Max/Mean: 0.0000 / 1.0000 / 0.2405
  ρ > 0.9 (solid): 22.7%
  ρ < 0.1 (void):  74.4%
  Grey (0.1–0.9):  2.8%   ← target: < 20%

  ✓ Polarization looks good.

Histogram saved: outputs/reports/cantilever_arm_density_histogram.png


Cell 3 — Isosurface extraction

In [ ]:
# Cell 3 — Extract isosurface
# ──────────────────────────────────────────────────────────────────────────
# CHECKPOINT: check face counts and surface area before continuing.
# Raw MC face count for 100×60×20 grid at threshold 0.3 is typically
# 200k-600k. If you see < 5k faces the threshold is too high.
# ──────────────────────────────────────────────────────────────────────────
import trimesh
import numpy as np

if solver_type == 'rust_voxel':
    from skimage.measure import marching_cubes

    vs = voxel_m  # voxel_size in metres

    # Pad with one layer of zeros on all sides.
    # Without this, MC leaves open boundary loops wherever the density array
    # terminates — the top/bottom faces and hole cylinders never close.
    # The zero padding forces MC to generate a complete closed surface.
    density_padded = np.pad(density, pad_width=1, mode='constant', constant_values=0.0)

    # ── Minimum feature size filter ───────────────────────────────────────
    # Morphological opening (erode → dilate) on the float density field.
    # Eliminates any solid feature thinner than MIN_FEATURE_MM before MC runs.
    # This is the primary tool for removing sub-nozzle-width internal struts
    # that threshold tuning cannot reach — they exist at full density (>0.9)
    # but are too thin to print.
    #
    # Kernel sizing:
    #   1.6mm @ 1mm voxels = 2-voxel radius → removes features < 2 voxels thick
    #   2.0mm @ 1mm voxels = 2-voxel radius (same — round up)
    #   0.0mm = disabled (raw density, no filtering)
    #
    # For FDM printing: MIN_FEATURE_MM = 2 × nozzle_diameter is a good start.
    # 0.4mm nozzle → 0.8mm min, but 1.6mm gives cleaner topology.
    # Raise to 2.4mm if Euler is still strongly negative after first run.
    MIN_FEATURE_MM = 1.6   # mm — set 0.0 to disable

    if MIN_FEATURE_MM > 0.0:
        from scipy.ndimage import grey_opening
        # Cubic morphological opening on the float density field.
        # Removes sub-feature voxels that fall below the kernel size.
        # Kernel is sized in voxels: round(MIN_FEATURE_MM / voxel_size_mm).
        # At 1mm voxels: 1.6mm → kernel 5³, 2.0mm → kernel 5³, 3.0mm → kernel 7³.
        # NOTE: internal structural ribs from the optimizer should NOT be
        # removed — MIN_FEATURE_MM should target surface fringe noise only
        # (recommended: 1.0–1.6mm at 1mm voxels). The optimizer's internal
        # topology is correct load-bearing structure, not a defect.
        kernel_voxels = max(1, round(MIN_FEATURE_MM / (vs * 1000.0)))
        kernel_size   = 2 * kernel_voxels + 1
        density_filtered = grey_opening(
            density_padded,
            size=kernel_size,
        ).astype(np.float32)
        n_removed = ((density_padded > DENSITY_THRESHOLD) &
                     (density_filtered <= DENSITY_THRESHOLD)).sum()
        print(f"  Morphological opening: kernel={kernel_size}³ voxels "
              f"({MIN_FEATURE_MM}mm), removed {n_removed:,} sub-feature voxels")
        density_mc = density_filtered
    else:
        density_mc = density_padded

    print(f"Running marching cubes: threshold={DENSITY_THRESHOLD}, "
          f"spacing=({vs},{vs},{vs}) m  (padded shape: {density_mc.shape})")

    verts, faces, normals, _ = marching_cubes(
        density_mc,
        level=DENSITY_THRESHOLD,
        spacing=(vs, vs, vs),
        allow_degenerate=False,
    )

    # The padding shifted all verts by one voxel — subtract it back
    verts = verts - vs

    # marching_cubes returns vertices in (iz, iy, ix) order.
    # Reorder to (ix, iy, iz) = (X, Y, Z) for correct STL orientation.
    verts = verts[:, [2, 1, 0]]
    normals = normals[:, [2, 1, 0]]

    # Flip Z so the loaded face (z_max) faces up in the exported STL
    verts[:, 2] = verts[:, 2].max() - verts[:, 2]
    normals[:, 2] = -normals[:, 2]

    # Convert metres → mm for STL export (Fusion 360 expects mm)
    verts_mm = verts * 1000.0

    print(f"  Raw MC:  {len(verts):,} verts, {len(faces):,} faces")
    print(f"  Bounds:  {verts_mm.min(axis=0)} → {verts_mm.max(axis=0)} mm")

    mesh_raw = trimesh.Trimesh(
        vertices=verts_mm,
        faces=faces,
        vertex_normals=normals,
        process=True,   # degenerate face removal, duplicate vertex merge
    )
    print(f"  After process=True: {len(mesh_raw.vertices):,} verts, "
          f"{len(mesh_raw.faces):,} faces")
    print(f"  Watertight (raw):   {mesh_raw.is_watertight}")

else:
    # FEniCSx tet boundary extraction (kept from original skeleton)
    print(f"Building tet boundary mesh at threshold={DENSITY_THRESHOLD}")
    solid_mask = rho_values > DENSITY_THRESHOLD
    solid_tets = tets[solid_mask]
    TET_FACE_COMBOS = [[0,1,2],[0,1,3],[0,2,3],[1,2,3]]
    TET_OPP_VERTEX  = [3, 2, 1, 0]
    all_faces_raw = solid_tets[:, TET_FACE_COMBOS].reshape(-1, 3)
    sorted_faces  = np.sort(all_faces_raw, axis=1)
    _, inverse, counts = np.unique(sorted_faces, axis=0,
                                   return_inverse=True, return_counts=True)
    boundary_mask = counts[inverse] == 1
    boundary_faces_final = []
    seen_keys = set()
    for fi_flat, is_bnd in enumerate(boundary_mask):
        if not is_bnd: continue
        tet_idx  = fi_flat // 4
        face_idx = fi_flat % 4
        tet = solid_tets[tet_idx]
        fc  = TET_FACE_COMBOS[face_idx]
        key = tuple(sorted_faces[fi_flat])
        if key in seen_keys: continue
        seen_keys.add(key)
        v0,v1,v2 = pts_mm[tet[fc[0]]],pts_mm[tet[fc[1]]],pts_mm[tet[fc[2]]]
        opp_v  = pts_mm[tet[TET_OPP_VERTEX[face_idx]]]
        normal = np.cross(v1-v0, v2-v0)
        if np.dot(normal, (v0+v1+v2)/3.0 - opp_v) > 0:
            boundary_faces_final.append([tet[fc[0]], tet[fc[1]], tet[fc[2]]])
        else:
            boundary_faces_final.append([tet[fc[0]], tet[fc[2]], tet[fc[1]]])
    boundary_faces_final = np.array(boundary_faces_final)
    mesh_raw = trimesh.Trimesh(
        vertices=pts_mm,
        faces=boundary_faces_final,
        process=True,
    )
    print(f"  Tet boundary: {len(mesh_raw.vertices):,} verts, "
          f"{len(mesh_raw.faces):,} faces")
    print(f"  Watertight (raw): {mesh_raw.is_watertight}")

Running marching cubes: threshold=0.45, spacing=(0.001,0.001,0.001) m  (padded shape: (77, 52, 102))
  Raw MC:  42,626 verts, 85,296 faces
  Bounds:  [-0.5500001  5.3051667 -0.5500001] → [74.55 49.55 99.55] mm
  After process=True: 42,626 verts, 85,296 faces
  Watertight (raw):   True


Cell 4 — Repair: largest component, fill holes, fix normals

In [ ]:
# Cell 4 — Mesh repair
# ──────────────────────────────────────────────────────────────────────────
# CHECKPOINT: island count, open edge count, watertight status.
# A well-polarized density field should give 1-3 components and few open edges.
# Many small floating islands → threshold too low, or density not polarized.
# ──────────────────────────────────────────────────────────────────────────
import numpy as np

mesh = mesh_raw

def open_edge_count(m):
    return len(trimesh.grouping.group_rows(m.edges_sorted, require_count=1))

print(f"Before repair:  {len(mesh.vertices):,} verts, "
      f"{len(mesh.faces):,} faces, "
      f"{open_edge_count(mesh)} open edges")

# ── Step 1: Remove islands ────────────────────────────────────────────────
if REMOVE_ISLANDS:
    components = mesh.split(only_watertight=False)
    vols = [abs(c.volume) for c in components]
    total_vol = max(sum(vols), 1.0)
    large = [c for c, v in zip(components, vols)
             if v >= ISLAND_VOL_FRAC * total_vol]
    if len(large) == 0:
        large = [max(components, key=lambda m: len(m.faces))]
    n_dropped = len(components) - len(large)
    if n_dropped > 0:
        dropped_vol = sum(v for c,v in zip(components,vols)
                         if v < ISLAND_VOL_FRAC * total_vol)
        print(f"  Islands: dropped {n_dropped} components "
              f"({dropped_vol:.1f} mm³ total), kept {len(large)}")
    else:
        print(f"  No islands to drop ({len(components)} components, "
              f"all \u2265{ISLAND_VOL_FRAC*100:.0f}% of total volume)")
    mesh = trimesh.util.concatenate(large) if len(large) > 1 else large[0]

# ── Step 2: Merge duplicate vertices ─────────────────────────────────────
mesh.merge_vertices()

# ── Step 3: Fill holes ────────────────────────────────────────────────────
trimesh.repair.fill_holes(mesh)
trimesh.repair.fill_holes(mesh)  # second pass catches holes exposed by first
trimesh.repair.fix_normals(mesh)

# If still not watertight, voxel remesh as last resort
# (guarantees closure at the cost of slightly softened edges)
if not mesh.is_watertight:
    print("  Not watertight after fill_holes — re-extracting at higher threshold...")
    from skimage.measure import marching_cubes
    fallback_threshold = DENSITY_THRESHOLD + 0.10
    verts2, faces2, normals2, _ = marching_cubes(
        density_padded,                              # padded — same as primary path
        level=fallback_threshold,
        spacing=(voxel_m, voxel_m, voxel_m),
        allow_degenerate=False,
    )
    # Remove padding offset
    verts2 = verts2 - voxel_m
    # Axis reorder: (iz,iy,ix) → (ix,iy,iz)
    verts2   = verts2[:, [2, 1, 0]]
    normals2 = normals2[:, [2, 1, 0]]
    # Z-flip: loaded face faces up
    verts2[:, 2]   = verts2[:, 2].max() - verts2[:, 2]
    normals2[:, 2] = -normals2[:, 2]
    mesh2 = trimesh.Trimesh(
        vertices=verts2 * 1000.0,
        faces=faces2,
        process=True,
    )
    trimesh.repair.fill_holes(mesh2)
    trimesh.repair.fill_holes(mesh2)
    trimesh.repair.fix_normals(mesh2)
    if mesh2.is_watertight:
        print(f"  Fallback threshold {fallback_threshold:.2f} gave watertight mesh "
            f"({len(mesh2.faces):,} faces)")
        mesh = mesh2
        DENSITY_THRESHOLD = fallback_threshold  # propagate to downstream cells
    else:
        print(f"  Fallback threshold {fallback_threshold:.2f} still not watertight "
              f"— {open_edge_count(mesh2)} open edges")

# ── Step 4: Fix normals / winding ─────────────────────────────────────────
trimesh.repair.fix_normals(mesh)

# ── Step 5: Drop degenerate/duplicate faces ───────────────────────────────
mesh.update_faces(mesh.nondegenerate_faces())
mesh.update_faces(mesh.unique_faces())
mesh.remove_unreferenced_vertices()

oe = open_edge_count(mesh)
print(f"\nAfter repair:   {len(mesh.vertices):,} verts, "
      f"{len(mesh.faces):,} faces, "
      f"{oe} open edges")
print(f"Watertight:     {mesh.is_watertight}")
print(f"Winding consistent: {mesh.is_winding_consistent}")
if oe > 0:
    print(f"\n  \u26a0 {oe} open edges remain after repair.")
    print("    Options: increase DENSITY_THRESHOLD slightly, or")
    print("    re-run notebook 04 with two-stage continuation.")

Before repair:  42,626 verts, 85,296 faces, 0 open edges
  Islands: dropped 3 components (728.7 mm³ total), kept 1

After repair:   41,084 verts, 82,220 faces, 0 open edges
Watertight:     True
Winding consistent: True


Cell 5 — Taubin smoothing and quadric decimation

In [6]:
# Cell 5 — Smoothing and simplification
# ──────────────────────────────────────────────────────────────────────────
# CHECKPOINT: verify watertight status is preserved after each step.
# Taubin is volume-preserving; Laplacian alone causes shrinkage on curved surfaces.
# ──────────────────────────────────────────────────────────────────────────

# ── Taubin smoothing ─────────────────────────────────────────────────────
if SMOOTH_ITERATIONS > 0:
    vol_before = abs(mesh.volume) if mesh.is_watertight else None
    trimesh.smoothing.filter_taubin(
        mesh,
        lamb=TAUBIN_LAMB,
        nu=TAUBIN_NU,
        iterations=SMOOTH_ITERATIONS,
    )
    trimesh.repair.fix_normals(mesh)
    oe_after = open_edge_count(mesh)
    vol_after = abs(mesh.volume) if mesh.is_watertight else None
    vol_change = ((vol_after - vol_before) / vol_before * 100
                  if vol_before and vol_after else None)
    print(f"After Taubin ×{SMOOTH_ITERATIONS}: "
          f"{len(mesh.vertices):,} verts, "
          f"{len(mesh.faces):,} faces, "
          f"{oe_after} open edges")
    if vol_change is not None:
        print(f"  Volume: {vol_before:.1f} → {vol_after:.1f} mm³ "
              f"(Δ={vol_change:+.2f}%)")
    if vol_change is not None and abs(vol_change) > 5.0:
        print("  ⚠ >5% volume change — check TAUBIN_NU sign")

# ── Quadric decimation ────────────────────────────────────────────────────
# trimesh.Trimesh.simplify_quadric_decimation delegates to open3d. Open3d
# isn't in the default dolfinx-env. If unavailable, skip decimation — the
# raw marching-cubes output is still a valid watertight printable STL,
# just larger. To enable quadric decimation for tighter STLs:
#     /dolfinx-env/bin/pip install open3d
try:
    import open3d  # noqa: F401
    _HAS_OPEN3D = True
except ImportError:
    _HAS_OPEN3D = False

n_raw = len(mesh.faces)
target_faces = max(int(n_raw * TARGET_FACE_FRACTION), MIN_FACES)
if not _HAS_OPEN3D:
    print(f"\nSkipping decimation: open3d not installed.")
    print(f"  Raw mesh: {n_raw:,} faces (target was {target_faces:,}).")
    print(f"  To enable: /dolfinx-env/bin/pip install open3d")
elif target_faces < n_raw:
    print(f"\nDecimating {n_raw:,} → {target_faces:,} faces "
          f"({TARGET_FACE_FRACTION*100:.0f}%)...")
    mesh = mesh.simplify_quadric_decimation(target_faces)
    # One more repair pass after decimation can introduce new holes
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    mesh.remove_unreferenced_vertices()
    oe_final = open_edge_count(mesh)
    print(f"After decimation: {len(mesh.vertices):,} verts, "
          f"{len(mesh.faces):,} faces, "
          f"{oe_final} open edges")
else:
    print(f"\nSkipping decimation (face count {n_raw:,} already ≤ target {target_faces:,})")

print(f"\nFinal mesh:")
print(f"  Verts:      {len(mesh.vertices):,}")
print(f"  Faces:      {len(mesh.faces):,}")
print(f"  Watertight: {mesh.is_watertight}")
print(f"  Winding:    {mesh.is_winding_consistent}")
if mesh.is_watertight:
    vol = abs(mesh.volume)
    # Expected: vf * domain_volume (mm³)
    if solver_type == 'rust_voxel':
        domain_vol = (voxel_m * nx * 1000) * (voxel_m * ny * 1000) * (voxel_m * nz * 1000)
        expected_vol = handoff.get('final_volume_frac', 0.45) * domain_vol
        print(f"  Volume:     {vol:.1f} mm³  (expected ≈{expected_vol:.0f} mm³)")
    else:
        print(f"  Volume:     {vol:.1f} mm³")


After Taubin ×10: 41,084 verts, 82,220 faces, 0 open edges
  Volume: 91336.8 → 89132.6 mm³ (Δ=-2.41%)

Skipping decimation: open3d not installed.
  Raw mesh: 82,220 faces (target was 24,666).
  To enable: /dolfinx-env/bin/pip install open3d

Final mesh:
  Verts:      41,084
  Faces:      82,220
  Watertight: True
  Winding:    True
  Volume:     89132.6 mm³  (expected ≈93750 mm³)


Cell 6 — Manifold verification and STL export

In [ ]:
# Cell 6 — Final verification and STL export
from pathlib import Path

stl_dir = Path("outputs/stl")
stl_dir.mkdir(parents=True, exist_ok=True)
stl_path = stl_dir / f"{part_name}_optimized.stl"
SMOOTH_ITERATIONS = 10
SMOOTH_LAMBDA = 0.5

# Final manifold check
# ── Mesh quality report ───────────────────────────────────────────────────
# Euler characteristic: V - E + F = 2 for a clean genus-0 closed manifold.
# Any other value indicates topological holes (handles/tunnels) in the mesh.
# trimesh.is_watertight can return False due to Euler ≠ 2 even with 0 open
# edges — the Euler number distinguishes topological defects from boundary gaps.
n_verts = len(mesh.vertices)
n_edges = len(mesh.edges_unique)
n_faces = len(mesh.faces)
euler   = n_verts - n_edges + n_faces
oe      = open_edge_count(mesh)

issues = []
if not mesh.is_watertight:         issues.append("not watertight")
if not mesh.is_winding_consistent: issues.append("winding inconsistent")
if oe > 0:                         issues.append(f"{oe} open edges")
if euler != 2:                     issues.append(f"Euler={euler} (expected 2 for genus-0)")

print("Mesh quality report:")
print(f"  Vertices:      {n_verts:,}")
print(f"  Edges:         {n_edges:,}")
print(f"  Faces:         {n_faces:,}")
print(f"  Open edges:    {oe}")
print(f"  Euler (V-E+F): {euler}  {'✓ genus-0' if euler == 2 else '⚠ topological defect'}")
print(f"  Watertight:    {mesh.is_watertight}")
print(f"  Winding:       {mesh.is_winding_consistent}")
if mesh.is_watertight:
    vol  = abs(mesh.volume)
    area = mesh.area
    print(f"  Volume:        {vol:.2f} mm³")
    print(f"  Surface area:  {area:.2f} mm²")
    print(f"  Bounds:        {mesh.bounds[0].round(2)} → {mesh.bounds[1].round(2)} mm")
    if solver_type == 'rust_voxel':
        domain_vol = (voxel_m * nx * 1000) * (voxel_m * ny * 1000) * (voxel_m * nz * 1000)
        vf_mesh    = vol / domain_vol
        vf_target  = handoff.get('final_volume_frac', 0.45)
        vf_ok      = abs(vf_mesh - vf_target) < 0.10
        print(f"  Vol fraction:  {vf_mesh:.3f}  (target {vf_target:.3f})  "
              f"{'✓' if vf_ok else '⚠ >10% off'}")

print()
if issues:
    print(f"⚠ Quality issues: {', '.join(issues)}")
    print("  The STL will still export but may need repair in Fusion 360.")
    print("  Suggestions:")
    print("    - Increase DENSITY_THRESHOLD (try +0.05, max 0.55)")
    print("    - Re-run Stage 4 with two-stage penal continuation")
    print("    - Increase SMOOTH_ITERATIONS (try 30)")
    if euler != 2:
        print(f"    - Euler={euler}: topological tunnels present — try raising "
              f"ISLAND_VOL_FRAC or DENSITY_THRESHOLD")
else:
    print("✓ All quality checks passed — mesh is print-ready")

# ── Laplacian smoothing ───────────────────────────────────────────────────
if SMOOTH_ITERATIONS > 0:
    import trimesh.smoothing
    mesh = trimesh.smoothing.filter_laplacian(
        mesh,
        lamb=SMOOTH_LAMBDA,
        iterations=SMOOTH_ITERATIONS,
        implicit_time_integration=False,
        volume_constraint=True,
    )
    print(f"Laplacian smooth: {SMOOTH_ITERATIONS} iters  λ={SMOOTH_LAMBDA}")
    print(f"  Watertight after smooth: {mesh.is_watertight}")
    
mesh.export(str(stl_path))
stl_size_kb = stl_path.stat().st_size / 1024

print(f"\nSTL exported:")
print(f"  Path:      {stl_path}")
print(f"  Size:      {stl_size_kb:.1f} KB")
print(f"  Triangles: {n_faces:,}")

# Quality checklist
print("\nQuality checklist (Fusion 360 import):")
print(f"  [{'✓' if mesh.is_watertight else '✗'}] is_watertight")
print(f"  [{'✓' if mesh.is_winding_consistent else '✗'}] is_winding_consistent")
print(f"  [{'✓' if oe == 0 else '✗'}] 0 open edges")
print(f"  [{'✓' if euler == 2 else '✗'}] Euler number = 2 (genus-0)")
print(f"  [{'✓' if n_faces < 100_000 else '⚠'}] triangles < 100k")
if mesh.is_watertight and solver_type == 'rust_voxel':
    vf_mesh   = abs(mesh.volume) / ((voxel_m*nx*1000) * (voxel_m*ny*1000) * (voxel_m*nz*1000))
    vf_target = handoff.get('final_volume_frac', 0.45)
    vf_ok     = abs(vf_mesh - vf_target) < 0.10
    print(f"  [{'✓' if vf_ok else '⚠'}] volume fraction ≈ target "
          f"(mesh: {vf_mesh:.3f}, target: {vf_target:.3f})")

NameError: name 'part_name' is not defined

Cell 6b — Boolean hole cutting (DISABLED)

In [8]:
# Cell 7 — Before/after render
if not RENDER_PLOTS:
    print("Skipping render — set RENDER_PLOTS=True after fixing display")
else:
    import pyvista as pv
    pv.OFF_SCREEN = True

    report_dir = Path('outputs/reports')
    original_stl = Path(f'outputs/meshes/{part_name}.stl')

    pl = pv.Plotter(shape=(1, 2), window_size=(1600, 600))
    if original_stl.exists():
        orig = pv.read(str(original_stl))
        pl.subplot(0, 0)
        pl.add_mesh(orig, color='lightsteelblue', show_edges=False, opacity=0.9)
        pl.add_text(f'Original\n{orig.n_cells:,} faces', font_size=10)
        pl.view_isometric()
    opt = pv.read(str(stl_path))
    pl.subplot(0, 1)
    pl.add_mesh(opt, color='coral', show_edges=False, opacity=0.9)
    pl.add_text(f'Optimized\n{opt.n_cells:,} faces', font_size=10)
    pl.view_isometric()
    compare_png = report_dir / f'{part_name}_before_after.png'
    pl.screenshot(str(compare_png))
    pl.close()
    print(f'Before/after: {compare_png}')


Skipping render — set RENDER_PLOTS=True after fixing display


Cell 8 — Pipeline completion record

In [ ]:
# Cell 8 — Write stage05 completion record
import json
from pathlib import Path
from datetime import datetime, timezone

stage_handoffs = {}
for stage_json in sorted(Path('outputs/meshes').glob(f'{part_name}_stage0*.json')):
    try:
        d = json.loads(stage_json.read_text())
        stage_handoffs[d['stage']] = d
    except Exception:
        pass

_s04 = stage_handoffs.get('04_simp', handoff)

record = {
    "stage":          "05_stl_export",
    "part_name":      part_name,
    "completed_at":   datetime.now(timezone.utc).isoformat(),
    "stl_path":       str(stl_path),
    "stl_size_kb":    round(stl_size_kb, 2),
    "mesh": {
        "vertices":   len(mesh.vertices),
        "triangles":  len(mesh.faces),
        "watertight": mesh.is_watertight,
        "euler":      euler,
        "open_edges": oe,
        "volume_mm3": round(abs(mesh.volume), 3) if mesh.is_watertight else None,
        "area_mm2":   round(mesh.area, 3),
    },
    'export_config': {
        'density_threshold':   DENSITY_THRESHOLD,
        'smooth_iterations':   SMOOTH_ITERATIONS,
        'taubin_lamb':         TAUBIN_LAMB,
        'taubin_nu':           TAUBIN_NU,
        'target_face_fraction': TARGET_FACE_FRACTION,
        'remove_islands':      REMOVE_ISLANDS,
    },
    'optimization': {
        'solver':          _s04.get('solver'),
        'volume_fraction': _s04.get('config', {}).get('volume_fraction'),
        'n_iterations':    _s04.get('n_iterations'),
        'converged':       _s04.get('converged'),
        'final_compliance':_s04.get('final_compliance'),
    },
}

record_path = Path('outputs/meshes') / f'{part_name}_stage05.json'
record_path.write_text(json.dumps(record, indent=2))

print('═' * 60)
print('  PIPELINE COMPLETE')
print('═' * 60)
print(f'  Part:          {part_name}')
print(f'  STL:           {stl_path}')
print(f'  Watertight:    {mesh.is_watertight}')
print(f'  Triangles:     {len(mesh.faces):,}')
if mesh.is_watertight:
    print(f'  Volume:        {abs(mesh.volume):.2f} mm³')
_vf = _s04.get('config', {}).get('volume_fraction')
if _vf:
    print(f'  Material saved: {(1 - _vf) * 100:.1f}%')
print('─' * 60)
print('  Stage durations:')
for stage, data in sorted(stage_handoffs.items()):
    dur = data.get('duration_s', '—')
    print(f'    {stage:<20} {dur}s')
print('═' * 60)
print(f'\n  Full record: {record_path}')


════════════════════════════════════════════════════════════
  PIPELINE COMPLETE
════════════════════════════════════════════════════════════
  Part:          cantilever_arm
  STL:           outputs/stl/cantilever_arm_optimized.stl
  Watertight:    True
  Triangles:     82,220
  Volume:        89132.55 mm³
  Material saved: 75.0%
────────────────────────────────────────────────────────────
  Stage durations:
    01_geometry          0.478s
    02_meshing           —s
    03_fea               —s
    04_simp              27931.534915397002s
    05_stl_export        —s
════════════════════════════════════════════════════════════

  Full record: outputs/meshes/cantilever_arm_stage05.json
